In [57]:
import os

import geopandas as gpd
import shapely

from pathlib import Path
import pandas as pd
from dotenv import load_dotenv
from afolu.assets.zones import ISO_TO_COUNTRY
from typing import Sequence

In [2]:
load_dotenv()

True

In [3]:
country_to_iso = {v: k for k, v in ISO_TO_COUNTRY.items()}

In [4]:
data_path = Path(os.environ["DATA_PATH"])
out_path = data_path / "generated"

ghsl_path = Path(os.environ["GHSL_PATH"])
amazonas_path = Path(os.environ["AMAZONAS_PATH"])

In [122]:
wanted_zones = [
    "BOL+Cobija",
    "BOL+Guayaramerín",
    "BOL+Riberalta",
    "BOL+Trinidad",
    "BRA+Abaetetuba",
    "BRA+Altamira",
    "BRA+Barcarena",
    "BRA+Belém",
    "BRA+Boa Vista",
    "BRA+Brasileia",
    "BRA+Cametá",
    "BRA+Igarapé-Miri",
    "BRA+Manaus",
    "BRA+Nova Ipixuna",
    "BRA+Paragominas",
    "BRA+Rio Branco",
    "BRA+Rio Preto da Eva",
    "BRA+Santarém",
    "BRA+Palmas",
    "BRA+Parauapebas",
    "BRA+São Luís",
    "BRA+São Gabriel da Cachoeira",
    "COL+Florencia",
    "COL+Milán - Caquetá",
    "COL+Mocoa",
    "COL+San José del Guaviare",
    "COL+Inírida",
    "COL+Leticia",
    "ECU+Macas-Morona-Santiago",
    "ECU+Lago Agrio",
    "ECU+Tena",
    "GUY+Georgetown",
    "GUY+Lethem",
    "PER+Maynas",
    "PER+Tambopata",
    "PER+San Martín",
    "PER+Coronel Portillo",
    "SUR+Nieuw Nickerie",
    "VEN+Atures",
]

zone_to_fua_name_map = {
    "PER+San Martín": "PER+Tarapoto",
    "PER+Coronel Portillo": "PER+Pucallpa",
    "VEN+Atures": "VEN+Puerto Ayacucho",
}

zone_to_polygon_map = {
    "BOL+Guayaramerín": [1962, 1950],
    "BRA+Barcarena": [1932, 1915, 694, 2324],
    "BRA+Cametá": [1914, 4211],
    "BRA+Igarapé-Miri": [2022],
    "BRA+Nova Ipixuna": [2361],
    "BRA+Rio Preto da Eva": [2160],
    "BRA+São Gabriel da Cachoeira": [2085],
    "COL+Milán - Caquetá": [3768],
    "COL+Mocoa": [1926, 9236],
    "COL+Inírida": [1958],
    "ECU+Macas-Morona-Santiago": [1976, 8857, 3791, 1437, 4948],
    "ECU+Lago Agrio": [1870],
    "PER+Maynas": [1871, 1881],
    "PER+Tambopata": [1856],
    "SUR+Nieuw Nickerie": [28_181],
    "GUY+Lethem": [10_055, 10_054]
}

In [123]:
df_fua = gpd.read_file(ghsl_path / "GHS_FUA_UCDB2015_GLOBE_R2019A_54009_1K_V1_0.gpkg")
df_uc = gpd.read_file(ghsl_path / "GHS_UCDB_GLOBE_R2024A.gpkg", layer="GHS_UCDB_THEME_GENERAL_CHARACTERISTICS_GLOBE_R2024A")
df_poly = gpd.read_file(amazonas_path / "final" / "final" / "polygons" / "split.gpkg")

In [11]:
import pandas as pd

pd.Series([1, 2, 3], index=["a", "b", "c"]).T

a    1
b    2
c    3
dtype: int64

In [124]:
def get_data_from_fua(country_iso: str, city_name: str, df_fua: gpd.GeoDataFrame) -> pd.Series | None:
    fua_list = df_fua.query(f"(Cntry_ISO == '{country_iso}') & (eFUA_name == '{city_name}')")
    if len(fua_list) > 1:
        err = f"Multiple FUA found for {country_iso} {city_name}"
        raise ValueError(err)

    if len(fua_list) == 1:
        return fua_list[["FUA_p_2015", "eFUA_name", "geometry"]].to_crs("EPSG:4326").rename(columns={"FUA_p_2015": "pop", "eFUA_name": "name"}).iloc[0]

    return None


def get_data_from_uc(country_iso: str, city_name: str, df_uc: gpd.GeoDataFrame) -> pd.Series | None:
    country_name = ISO_TO_COUNTRY[country_iso]

    uc_list = df_uc.query(f"(GC_CNT_GAD_2025 == '{country_name}') & (GC_UCN_MAI_2025 == '{city_name}')")
    if len(uc_list) > 1:
        err = f"Multiple UC found for {country_iso} {city_name}"
        raise ValueError(err)

    if len(uc_list) == 1:
        return uc_list[["GC_POP_TOT_2025", "GC_UCN_MAI_2025", "geometry"]].to_crs("EPSG:4326").rename(columns={"GC_POP_TOT_2025": "pop", "GC_UCN_MAI_2025": "name"}).iloc[0]

    return None


def get_data_from_polygons(polygon_ids: Sequence[int], df_poly: gpd.GeoDataFrame) -> pd.Series | None:
    poly_list = df_poly.query(f"polygon_id.isin(@polygon_ids)")

    names_filtered = poly_list.query("~name.str.startswith('Cerca')")["name"]

    return pd.Series({
        "pop": poly_list["pop_2020"].sum(),
        "geometry":poly_list["geometry"].to_crs("EPSG:4326").union_all(),
        "name": names_filtered.str.cat(sep="+")
    })


def find_pop_and_geometry(
    zone: str,
    *,
    df_fua: gpd.GeoDataFrame,
    df_uc: gpd.GeoDataFrame,
    df_poly: gpd.GeoDataFrame,
    zone_to_fua_name_map: dict[str, str] | None=None,
    zone_to_polygon_map: dict[str, list[int]] | None=None
) -> pd.Series | None:
    if zone in zone_to_polygon_map:
        return get_data_from_polygons(zone_to_polygon_map[zone], df_poly)

    if zone in zone_to_fua_name_map:
        zone = zone_to_fua_name_map[zone]

    country_iso, city_name = zone.split("+")

    data_fua = get_data_from_fua(country_iso, city_name, df_fua)
    if data_fua is not None:
        return data_fua

    data_uc = get_data_from_uc(country_iso, city_name, df_uc)
    if data_uc is not None:
        return data_uc

    return None

In [125]:
rows = []
for zone in wanted_zones:
    res = find_pop_and_geometry(
        zone,
        df_fua=df_fua,
        df_uc=df_uc,
        df_poly=df_poly,
        zone_to_fua_name_map=zone_to_fua_name_map,
        zone_to_polygon_map=zone_to_polygon_map
    )

    country, name = zone.split("+")

    if res is not None:
        res["country"] = ISO_TO_COUNTRY[country]
        res["requested"] = name
        rows.append(res)
    else:
        pass
df_pop = (
    gpd.GeoDataFrame(pd.concat(rows, ignore_index=True, axis=1).T, crs="EPSG:4326", geometry="geometry")
    .assign(area=lambda df: df["geometry"].to_crs("ESRI:102033").area / 1e6, pop=lambda df: df["pop"].round(0).astype(int))
    .rename(columns={
        "pop": "Población",
        "country": "País",
        "requested": "Nombre solicitado",
        "area": "Área (km²)",
        "name": "Nombre de la ciudad utilizada",
    })
    .sort_values(["País", "Nombre solicitado"])
)

In [128]:
temp = df_pop.drop(columns=["geometry"])[["País", "Nombre solicitado", "Nombre de la ciudad utilizada", "Población", "Área (km²)"]]
with pd.ExcelWriter("./test.xlsx") as writer:
    temp.query("Población >= 100_000").to_excel(writer, sheet_name="Población > 100k", index=False)
    temp.query("(Población >= 50_000) & (Población < 100_000)").to_excel(writer, sheet_name="Población entre 50k y 100k", index=False)
    temp.query("Población < 50_000").to_excel(writer, sheet_name="Población < 50k", index=False)

In [127]:
df_pop.query("Población >= 100_000")

,Población,Nombre de la ciudad utilizada,geometry,País,Nombre solicitado,Área (km²)
3,111412,Trinidad,"MULTIPOLYGON (((-64.90331 -14.78254, -64.89312...",Bolivia,Trinidad,51.698159
6,106017,São Francisco+Barcarena+Murucupi,"MULTIPOLYGON (((-48.75282 -1.57717, -48.7628 -...",Brasil,Barcarena,56.618314
7,2174968,Belém,"MULTIPOLYGON (((-48.26944 -1.23745, -48.25946 ...",Brasil,Belém,583.076503
8,339308,Boa Vista,"MULTIPOLYGON (((-60.68248 2.88777, -60.67249 2...",Brasil,Boa Vista,173.833573
11,2080861,Manaus,"MULTIPOLYGON (((-60.02423 -2.91204, -60.01424 ...",Brasil,Manaus,449.990131
17,179154,Palmas,"MULTIPOLYGON (((-48.30175 -10.15425, -48.29168...",Brasil,Palmas,91.419975
18,211964,Parauapebas,"MULTIPOLYGON (((-49.88787 -5.99717, -49.87786 ...",Brasil,Parauapebas,69.542515
14,378148,Rio Branco,"MULTIPOLYGON (((-67.89196 -9.86951, -67.88189 ...",Brasil,Rio Branco,265.320798
16,226715,Santarém,"MULTIPOLYGON (((-54.69828 -2.41849, -54.6883 -...",Brasil,Santarém,199.656707
19,1455771,São Luís,"MULTIPOLYGON (((-44.09754 -2.49131, -44.08756 ...",Brasil,São Luís,551.293533
